In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import zipfile
import os
import json

In [2]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print("Device:", device)

Device: cuda


In [4]:
with zipfile.ZipFile('/content/archive.zip', 'r') as zf:
    zf.extractall('/content/')

In [6]:
print(os.listdir('/content/'))

['.config', 'test', 'train', 'archive.zip', 'sports.csv', 'valid', 'EfficientNetB0-100-(224 X 224)- 98.40.h5', 'sample_data']


In [8]:
transform_data = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])



In [9]:
train_data = datasets.ImageFolder('/content/train', transform=transform_data)
test_data = datasets.ImageFolder('/content/test', transform=transform_data)

In [10]:
print("Train:", len(train_data))
print("Test:", len(test_data))
print("Classes:", len(train_data.classes))

print(train_data.classes == test_data.classes)

Train: 13492
Test: 500
Classes: 100
True


In [11]:
train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False
)

In [12]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

Images shape: torch.Size([64, 3, 128, 128])
Labels shape: torch.Size([64])


In [13]:
model = models.resnet18(weights='DEFAULT')
model.fc = nn.Linear(model.fc.in_features, 100)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 168MB/s]


In [14]:
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [15]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [16]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 1.8610
Epoch 2, Loss: 0.8132
Epoch 3, Loss: 0.4571
Epoch 4, Loss: 0.2835
Epoch 5, Loss: 0.1845
Epoch 6, Loss: 0.1407
Epoch 7, Loss: 0.1929
Epoch 8, Loss: 0.1562
Epoch 9, Loss: 0.1468
Epoch 10, Loss: 0.1064


In [17]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 77.80%


In [18]:
torch.save(model.state_dict(), 'model.pth')